# Notebook 1 — Data Exploration
**Port Intelligence BI Training Program**

In this notebook you will explore the `port_calls` dataset (76,000 rows, Jan 2024–Dec 2025).
You will build intuition about the waiting-time distribution, port differences, vessel patterns,
and temporal rhythms before any modelling begins.

---

## 1  Setup
Import standard libraries and load the parquet dataset.

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Consistent style across all notebooks
sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.05)
plt.rcParams['figure.dpi'] = 110

# Load dataset
df = pd.read_parquet('../../data/port_calls.parquet')

# Ensure datetime columns are parsed
for col in ['eta_planned', 'ata_actual', 'atb', 'etd', 'atd_actual', 'created_date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

print('Dataset loaded successfully.')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

## 2  Dataset Overview
Inspect shape, data types, a sample of rows, and null counts.

In [ ]:
print('=== Shape ===')
print(f'Rows: {df.shape[0]:,}   Columns: {df.shape[1]}')

print('\n=== Column dtypes ===')
print(df.dtypes.to_string())

print('\n=== First 5 rows ===')
display(df.head())

print('\n=== Null counts ===')
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if null_counts.empty:
    print('No null values found — dataset is complete.')
else:
    print(null_counts.to_string())

print('\n=== Numeric summary ===')
display(df[['waiting_anchor_hours', 'waiting_berth_hours', 'cargo_tons',
            'teu_loaded', 'teu_discharged', 'weather_wind_knots',
            'berth_competition']].describe().round(2))

## 3  Waiting Time Distribution
The target variable for Model 1 is `waiting_anchor_hours`.
We plot its histogram and mark key percentiles (P50, P80, P95, P99).

In [ ]:
wait = df['waiting_anchor_hours'].clip(upper=50)

# Compute percentiles
pcts = {50: None, 80: None, 95: None, 99: None}
for p in pcts:
    pcts[p] = np.percentile(df['waiting_anchor_hours'], p)

print('Waiting time percentiles (raw, no clip):')
for p, v in pcts.items():
    print(f'  P{p:>2d} = {v:.1f} h')

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(wait, bins=80, color='steelblue', edgecolor='white', linewidth=0.3, alpha=0.85)

colors = {50: '#2ca02c', 80: '#ff7f0e', 95: '#d62728', 99: '#9467bd'}
for p, v in pcts.items():
    clipped_v = min(v, 50)
    ax.axvline(clipped_v, color=colors[p], linestyle='--', linewidth=1.6,
               label=f'P{p} = {v:.1f} h')

ax.set_xlabel('Waiting time at anchor (hours, clipped at 50 h)')
ax.set_ylabel('Number of port calls')
ax.set_title('Distribution of Anchor Waiting Times — All Calls')
ax.legend()
plt.tight_layout()
plt.show()

pct_zero = (df['waiting_anchor_hours'] == 0).mean() * 100
print(f'\nCalls with zero waiting time: {pct_zero:.1f}%')
print(f'Mean waiting time: {df["waiting_anchor_hours"].mean():.2f} h')
print(f'Std  waiting time: {df["waiting_anchor_hours"].std():.2f} h')

## 4  Port Comparison
Compare call volumes and waiting-time distributions between Haifa and Ashdod.

In [ ]:
port_counts = df['port_name'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart: call counts by port
axes[0].bar(port_counts.index, port_counts.values,
            color=['#1f77b4', '#ff7f0e'], edgecolor='white')
axes[0].set_xlabel('Port')
axes[0].set_ylabel('Number of port calls')
axes[0].set_title('Port Call Counts by Port')
for i, (port, count) in enumerate(port_counts.items()):
    axes[0].text(i, count + 200, f'{count:,}', ha='center', fontsize=10, fontweight='bold')

# Box plot: waiting time by port
port_groups = [df.loc[df['port_name'] == p, 'waiting_anchor_hours'].clip(upper=60)
               for p in port_counts.index]
bp = axes[1].boxplot(port_groups, labels=port_counts.index,
                     patch_artist=True, notch=False,
                     medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], ['#1f77b4', '#ff7f0e']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_xlabel('Port')
axes[1].set_ylabel('Waiting time at anchor (h, clipped 60 h)')
axes[1].set_title('Waiting Time Distribution by Port')

plt.tight_layout()
plt.show()

print('Summary statistics by port:')
print(df.groupby('port_name')['waiting_anchor_hours']
        .describe(percentiles=[.5, .8, .95])
        .round(2)
        .to_string())

## 5  Vessel Type Analysis
Examine which vessel types dominate the dataset and whether vessel type
influences average waiting time across ports.

In [ ]:
vessel_counts = df['vessel_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: vessel type counts
axes[0].bar(vessel_counts.index, vessel_counts.values,
            color=sns.color_palette('tab10', len(vessel_counts)))
axes[0].set_xlabel('Vessel type')
axes[0].set_ylabel('Count')
axes[0].set_title('Port Calls by Vessel Type')
axes[0].tick_params(axis='x', rotation=15)
for i, (vt, cnt) in enumerate(vessel_counts.items()):
    axes[0].text(i, cnt + 100, f'{cnt:,}', ha='center', fontsize=9)

# Heatmap: avg wait by (vessel_type, port_name)
pivot = (df.groupby(['vessel_type', 'port_name'])['waiting_anchor_hours']
           .mean()
           .unstack('port_name')
           .round(2))
sns.heatmap(pivot, ax=axes[1], annot=True, fmt='.1f',
            cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Avg waiting hours'})
axes[1].set_title('Avg Anchor Wait (h) by Vessel Type & Port')
axes[1].set_xlabel('Port')
axes[1].set_ylabel('Vessel type')

plt.tight_layout()
plt.show()

print('Average waiting time by vessel type:')
print(df.groupby('vessel_type')['waiting_anchor_hours'].mean().sort_values(ascending=False).round(2).to_string())

## 6  Temporal Patterns
Understand the flow of arrivals over calendar time and intraday/weekly cycles.

In [ ]:
df['date'] = df['ata_actual'].dt.date
df['hour'] = df['ata_actual'].dt.hour
df['dow']  = df['ata_actual'].dt.dayofweek   # 0=Mon

daily_counts = df.groupby('date').size().reset_index(name='calls')
daily_counts['date'] = pd.to_datetime(daily_counts['date'])

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Line chart: daily call counts
axes[0].plot(daily_counts['date'], daily_counts['calls'],
             color='steelblue', linewidth=0.8, alpha=0.7)
# 30-day rolling mean
rolling = daily_counts.set_index('date')['calls'].rolling('30D').mean()
axes[0].plot(rolling.index, rolling.values,
             color='#d62728', linewidth=2, label='30-day rolling mean')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Daily arrivals')
axes[0].set_title('Daily Port Call Volume (Jan 2024 – Dec 2025)')
axes[0].legend()

# Heatmap: arrivals by (day_of_week, hour)
dow_hour = df.groupby(['dow', 'hour']).size().unstack('hour').fillna(0)
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_hour.index = dow_labels
sns.heatmap(dow_hour, ax=axes[1], cmap='Blues',
            cbar_kws={'label': 'Arrival count'},
            linewidths=0.2, linecolor='white')
axes[1].set_title('Arrival Heatmap: Day-of-Week vs Hour-of-Day')
axes[1].set_xlabel('Hour of day')
axes[1].set_ylabel('Day of week')

plt.tight_layout()
plt.show()

busiest_hour = df.groupby('hour').size().idxmax()
busiest_day  = df.groupby('dow').size().idxmax()
print(f'Busiest hour of day: {busiest_hour}:00')
print(f'Busiest day of week: {["Mon","Tue","Wed","Thu","Fri","Sat","Sun"][busiest_day]}')

## 7  Berth Competition Impact
`berth_competition` is a causal driver of waiting time.
We visualise the relationship and quantify it with Pearson correlation.

In [ ]:
# Sample 5,000 rows for readability
sample = df.sample(5000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: berth_competition vs waiting_anchor_hours
scatter_kw = dict(alpha=0.3, s=12, edgecolors='none')
colors_port = sample['port_name'].map({'Haifa': '#1f77b4', 'Ashdod': '#ff7f0e'})
axes[0].scatter(sample['berth_competition'],
                sample['waiting_anchor_hours'].clip(upper=60),
                c=colors_port, **scatter_kw)
# Add legend proxies
from matplotlib.lines import Line2D
handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=8, label=p)
           for p, c in [('Haifa','#1f77b4'), ('Ashdod','#ff7f0e')]]
axes[0].legend(handles=handles)
axes[0].set_xlabel('Berth competition score')
axes[0].set_ylabel('Waiting time at anchor (h, clipped 60 h)')
axes[0].set_title('Berth Competition vs Waiting Time (n=5,000 sample)')

# Box plot: waiting time binned by competition quartile
df['competition_quartile'] = pd.qcut(df['berth_competition'], 4,
                                     labels=['Q1\n(low)', 'Q2', 'Q3', 'Q4\n(high)'])
groups = [df.loc[df['competition_quartile'] == q, 'waiting_anchor_hours'].clip(upper=60)
          for q in ['Q1\n(low)', 'Q2', 'Q3', 'Q4\n(high)']]
bp = axes[1].boxplot(groups, labels=['Q1\n(low)', 'Q2', 'Q3', 'Q4\n(high)'],
                     patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], sns.color_palette('YlOrRd', 4)):
    patch.set_facecolor(color)
axes[1].set_xlabel('Berth competition quartile')
axes[1].set_ylabel('Waiting time at anchor (h, clipped 60 h)')
axes[1].set_title('Waiting Time by Competition Quartile')

plt.tight_layout()
plt.show()

# Pearson correlation
corr = df['berth_competition'].corr(df['waiting_anchor_hours'])
print(f'Pearson r(berth_competition, waiting_anchor_hours) = {corr:.4f}')

# Also check weather
corr_wind = df['weather_wind_knots'].corr(df['waiting_anchor_hours'])
print(f'Pearson r(weather_wind_knots, waiting_anchor_hours) = {corr_wind:.4f}')

## 8  Exercises

Complete the code cells below. Replace each `# YOUR CODE HERE` comment
with working Python.

**Exercise 1** — What is the median waiting time for CONTAINER vessels arriving at Haifa on weekends?

**Exercise 2** — Which `service_line` has the highest average `waiting_anchor_hours`?

**Exercise 3** — Plot a histogram of `cargo_tons` separately for Haifa and Ashdod on the same axes.

In [ ]:
# Exercise 1
# Filter to CONTAINER vessels at Haifa arriving on weekends (day_of_week >= 5)

df['day_of_week'] = df['ata_actual'].dt.dayofweek

mask = (
    # YOUR CODE HERE: add conditions for vessel_type, port_name, and weekend
    (df['vessel_type'] == '???') &
    (df['port_name'] == '???') &
    (df['day_of_week'] >= ???)
)
median_wait = df.loc[mask, 'waiting_anchor_hours'].median()
print(f'Median weekend container wait at Haifa: {median_wait:.2f} h')

In [ ]:
# Exercise 2
# Group by service_line and compute mean waiting_anchor_hours, then find the maximum

svc_wait = df.groupby('???')['waiting_anchor_hours'].???()  # YOUR CODE HERE
busiest_svc = svc_wait.idxmax()
print(f'Service line with highest avg wait: {busiest_svc} ({svc_wait[busiest_svc]:.2f} h)')
print(svc_wait.sort_values(ascending=False).round(2).to_string())

In [ ]:
# Exercise 3
# Plot overlapping histograms of cargo_tons for Haifa and Ashdod

fig, ax = plt.subplots(figsize=(10, 4))

for port, color in [('Haifa', '#1f77b4'), ('Ashdod', '#ff7f0e')]:
    subset = df.loc[df['port_name'] == port, '???']  # YOUR CODE HERE
    ax.hist(subset, bins=60, alpha=0.55, color=color, label=port, edgecolor='none')

ax.set_xlabel('Cargo tons')
ax.set_ylabel('Count')
ax.set_title('Cargo Volume Distribution by Port')
ax.legend()
plt.tight_layout()
plt.show()